# Master Regular Expressions (re) in Python: From Beginner to Advanced
### Real World Project: Customer Support & Security Log Parser

In this project, we process unstructured server/support logs into structured analytical datasets using Python's built in `re` module.

In [1]:
import re
import pandas as pd

# Verify dataset
with open('support_logs.txt', 'r') as f:
    raw_lines = f.readlines()

print(f'Total log entries loaded: {len(raw_lines)}')
print('Sample entry:\n', raw_lines[0])

Total log entries loaded: 8
Sample entry:
 [2026-08-10 09:15:22] [INFO] User ID: USR-9821, Email: sarah.connor@cyberdyne.org, Phone: +1-555-019-2834, Message: Order #ORD-45892 payment processed successfully. Amount: $149.99



## 1.Basic Pattern Matching & Token Extraction
- `re.findall()`: Extract simple patterns like Order IDs (`ORD-XXXXX`), IP addresses, and currency amounts.
- Character classes (`\d`, `\w`, `\s`) and quantifiers (`+`, `*`, `{n}`).

In [2]:
# 1. Extract all Order IDs
all_text = ''.join(raw_lines)
order_ids = re.findall(r'ORD-\d+', all_text)
print('Extracted Order IDs:', order_ids)

# 2. Extract all IP Addresses
ip_addresses = re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', all_text)
print('Extracted IP Addresses:', ip_addresses)

# 3. Extract all monetary amounts
amounts = re.findall(r'\$\d+(?:\.\d{2})?', all_text)
print('Extracted Amounts:', amounts)

Extracted Order IDs: ['ORD-45892', 'ORD-11029', 'ORD-89301', 'ORD-66712', 'ORD-99182']
Extracted IP Addresses: ['192.168.1.45', '10.0.4.12']
Extracted Amounts: ['$149.99', '$49.00']


## 2. Named Capture Groups & Structured Parsing
- Use `(?P<group_name>pattern)` to extract full records directly into clean key value dictionaries.

In [3]:
# Master log parser with Named Groups
log_pattern = re.compile(
    r'^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\]\s+'
    r'\[(?P<log_level>INFO|WARN|ERROR)\]\s+'
    r'User ID:\s+(?P<user_id>USR-\d+),\s+'
    r'Email:\s+(?P<email>[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+),\s+'
    r'Phone:\s+(?P<phone>[\+\d\s\(\)-]+),\s+'
    r'Message:\s+(?P<message>.*)$'
)

order_pattern = re.compile(r'ORD-\d+')
amount_pattern = re.compile(r'\$\d+(?:\.\d{2})?')
ip_pattern = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')

records = []
for line in raw_lines:
    match = log_pattern.match(line.strip())
    if match:
        item = match.groupdict()
        msg = item['message']
        
        # Sub-entity extraction
        ord_m = order_pattern.search(msg)
        amt_m = amount_pattern.search(msg)
        ip_m = ip_pattern.search(msg)
        
        item['order_id'] = ord_m.group(0) if ord_m else None
        item['amount'] = amt_m.group(0) if amt_m else None
        item['ip_address'] = ip_m.group(0) if ip_m else None
        
        records.append(item)

df = pd.DataFrame(records)
df.head()

,timestamp,log_level,user_id,email,phone,message,order_id,amount,ip_address
0,2026-08-10 09:15:22,INFO,USR-9821,sarah.connor@cyberdyne.org,+1-555-019-2834,Order #ORD-45892 payment processed successfull...,ORD-45892,$149.99,None
1,2026-08-10 09:16:05,ERROR,USR-1044,john_doe99@gmail.com,(555) 234-5678,Refund request failed for Order #ORD-11029. Er...,ORD-11029,None,None
2,2026-08-10 09:20:41,WARN,USR-7823,alex.smith+support@techcorp.io,+92-300-1234567,Duplicate attempt on Order #ORD-89301.,ORD-89301,None,None
3,2026-08-10 09:22:15,INFO,USR-3319,maria_garcia@freemail.net,555-876-5432,Order #ORD-66712 shipped via DHL. Tracking lin...,ORD-66712,None,None
4,2026-08-10 09:25:30,ERROR,USR-5590,bruce.wayne@wayne-ent.co.uk,+44 20 7946 0912,Account locked after 3 failed password attempt...,None,None,192.168.1.45


## 3. Lookarounds & Data Privacy Masking (re.sub)
- Lookaheads `(?=...)` & Lookbehinds `(?<=...)` for context sensitive validation.
- PII Masking: Masking emails and normalizing inconsistent international phone numbers.

In [4]:
# Email PII Masking: keep first 2 chars + mask rest before @
def mask_email(email):
    return re.sub(r'(^[^@]{2})([^@]+)(@.*)', r'\1***\3', email)

# Normalize Phone numbers to standard digits
def clean_phone(phone):
    return re.sub(r'[^\d+]', '', phone)

# Extract Tracking URL only if preceded by 'Tracking link: '
def extract_tracking_url(msg):
    match = re.search(r'(?<=Tracking link:\s)(https?://[^\s]+)', msg)
    return match.group(0) if match else None

df['masked_email'] = df['email'].apply(mask_email)
df['clean_phone'] = df['phone'].apply(clean_phone)
df['tracking_url'] = df['message'].apply(extract_tracking_url)

# Final Clean Data View
df[['timestamp', 'log_level', 'user_id', 'masked_email', 'clean_phone', 'order_id', 'amount', 'tracking_url']]

,timestamp,log_level,user_id,masked_email,clean_phone,order_id,amount,tracking_url
0,2026-08-10 09:15:22,INFO,USR-9821,sa***@cyberdyne.org,+15550192834,ORD-45892,$149.99,None
1,2026-08-10 09:16:05,ERROR,USR-1044,jo***@gmail.com,5552345678,ORD-11029,None,None
2,2026-08-10 09:20:41,WARN,USR-7823,al***@techcorp.io,+923001234567,ORD-89301,None,None
3,2026-08-10 09:22:15,INFO,USR-3319,ma***@freemail.net,5558765432,ORD-66712,None,https://track.dhl.com/v1/shipments/PK99281
4,2026-08-10 09:25:30,ERROR,USR-5590,br***@wayne-ent.co.uk,+442079460912,None,None,None
5,2026-08-10 09:30:12,INFO,USR-4421,co***@innovate.pk,+923337654321,ORD-99182,$49.00,None
6,2026-08-10 09:35:45,WARN,USR-6102,na***@avengers.com,0213456789,None,None,None
7,2026-08-10 09:40:00,INFO,USR-8833,cl***@dailyplanet.com,+18005550199,None,None,None


In [5]:
# Export clean structured dataframe to CSV
df.to_csv('cleaned_support_logs.csv', index=False)